In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load league-wide game data and filter to MIA home games
data = pd.read_csv('../../../data/league_weather_2021_2025.csv')
mia = data[data['home_team'] == 'MIA'].copy()

# Create wind speed quintile bins (reproduces the 5 heatmap buckets)
mia['wspd_bin'] = pd.qcut(mia['wspd_mph'], q=5, duplicates='drop')

print("Wind speed bins and game counts:")
print(mia['wspd_bin'].value_counts().sort_index())
print(f"\nTotal MIA home games: {len(mia)}")

In [ ]:
# ============================================================
# Weather Profile by Wind Speed Bucket
# ============================================================

weather_vars = {
    'temp_f': {'label': 'Temperature (°F)', 'color': '#d62728'},
    'rhum':   {'label': 'Humidity (%)',      'color': '#1f77b4'},
    'pres':   {'label': 'Pressure (hPa)',    'color': '#9467bd'},
}

bin_order = mia['wspd_bin'].cat.categories
bin_strs = [str(b) for b in bin_order]
tick_labels = [s.replace('(', '').replace(']', '').replace(', ', '–') for s in bin_strs]
x = np.arange(len(bin_order))

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=False)

for ax, (col, info) in zip(axes, weather_vars.items()):
    grouped = mia.groupby('wspd_bin', observed=True)[col]
    means = grouped.mean().reindex(bin_order)
    sems = grouped.sem().reindex(bin_order)

    bars = ax.bar(x, means, yerr=sems, capsize=4,
                  color=info['color'], alpha=0.75, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=30, ha='right', fontsize=9)
    ax.set_title(info['label'], fontsize=13)
    ax.set_xlabel('Wind Speed Bin (mph)', fontsize=10)
    ax.set_ylabel(info['label'], fontsize=10)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

    for xi, m in zip(x, means):
        ax.text(xi, m + sems.iloc[xi] + 0.3, f'{m:.1f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Typical Weather Conditions by Wind Speed Quintile — loanDepot Park', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Wind Direction Distribution on High vs. Low Wind Days
# ============================================================

# Split into high wind (top quintile) vs. rest
top_bin = mia['wspd_bin'].cat.categories[-1]
mia['high_wind'] = mia['wspd_bin'] == top_bin

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, subset) in zip(axes, [('Low–Moderate Wind', ~mia['high_wind']),
                                        ('High Wind (Top Quintile)', mia['high_wind'])]):
    dir_counts = mia.loc[subset, 'wind_dir_bucket'].value_counts()
    compass = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    dir_counts = dir_counts.reindex(compass, fill_value=0)
    ax.bar(dir_counts.index, dir_counts.values, color='steelblue', alpha=0.75, edgecolor='black', linewidth=0.5)
    ax.set_title(label, fontsize=13)
    ax.set_xlabel('Wind Direction', fontsize=11)
    ax.set_ylabel('Number of Games', fontsize=11)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle('Wind Direction: High Wind vs. Low–Moderate Wind Days — loanDepot Park', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary Statistics Table
# ============================================================

summary = mia.groupby('wspd_bin', observed=True).agg(
    games=('wspd_mph', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    rhum_mean=('rhum', 'mean'),
    rhum_std=('rhum', 'std'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
    away_runs_mean=('away_runs_scored', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
).round(2)

summary.index.name = 'Wind Speed Bin (mph)'
summary.columns = [
    'Games', 'Temp Mean', 'Temp Std',
    'Humidity Mean', 'Humidity Std',
    'Pressure Mean', 'Pressure Std',
    'Away Runs Mean', 'Strikeouts Mean',
]
summary